# 14 - Reset Development or Test data

Deletes all current people-counter rows while preserving the 20 Delta table definitions, three committed views, and required empty registration-lock seed. This destructive operator tool is for isolated Development and Test Lakehouses only.

**Never deploy or run this notebook in Production.** Complete the maintenance stop gate in `README.md` section 12.2 before running it. The Fabric capacity remains active, but no people-counter application job may be queued or running.

**After importing into Fabric:** On the configuration code cell, select **... -> Toggle parameter cell** and confirm the parameter indicator. Attach and pin the intended Development or Test Lakehouse as this notebook's default Lakehouse. Verify the pinned Lakehouse in the explorer; do not rely only on workspace or notebook names.

Set `ENVIRONMENT` to `dev` or `test`. Set `CONFIRM_RESET` exactly to `RESET <ENVIRONMENT> <target>`, where `<target>` is `<TABLE_PREFIX>` when `DATABASE` is empty or `<DATABASE>.<TABLE_PREFIX>` otherwise. For example: `RESET TEST people_counter`. Leave the saved parameter default empty and pass the confirmation only for an approved manual run.

In [ ]:
ENVIRONMENT = ""
CONFIRM_RESET = ""
DATABASE = ""
TABLE_PREFIX = "people_counter"

In [ ]:
from datetime import datetime, timezone
import json
import re

import notebookutils
from delta.tables import DeltaTable
from pyspark.sql import SparkSession


IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
RESET_ORDER = (
    "gold_dim_model_config",
    "gold_dim_video",
    "gold_dim_location",
    "gold_dim_camera",
    "gold_dim_time",
    "gold_dim_date",
    "gold_operations_hour",
    "gold_video",
    "gold_flow_hour",
    "gold_flow_minute",
    "processing_benchmarks",
    "reconciliation_findings",
    "line_count_attempts",
    "telemetry_attempts",
    "replay_requests",
    "video_attempts",
    "video_work",
    "event_receipts",
    "dispatcher_leases",
    "registration_leases",
)
COMMITTED_VIEWS = (
    "telemetry_committed",
    "line_counts_committed",
    "runs_committed",
)


def identifier(value: str, name: str, *, allow_empty: bool = False) -> str:
    if allow_empty and not value:
        return ""
    if IDENTIFIER.fullmatch(value) is None:
        raise ValueError(f"{name} is not a valid SQL identifier: {value!r}")
    return value


environment = ENVIRONMENT.strip().lower()
if environment not in {"dev", "test"}:
    raise ValueError("ENVIRONMENT must be dev or test; Production reset is prohibited")

database = identifier(DATABASE.strip(), "DATABASE", allow_empty=True)
prefix = identifier(TABLE_PREFIX.strip(), "TABLE_PREFIX")
target = f"{database}.{prefix}" if database else prefix
expected_confirmation = f"RESET {environment.upper()} {target}"
if CONFIRM_RESET != expected_confirmation:
    raise ValueError(
        "Reset not confirmed. Set CONFIRM_RESET exactly to "
        f"{expected_confirmation!r} after verifying the pinned Lakehouse"
    )

spark_candidate = globals().get("spark")
if not isinstance(spark_candidate, SparkSession):
    raise RuntimeError("Attach the target Fabric Lakehouse and start a Spark session")
spark_session = spark_candidate
spark_session.conf.set("spark.sql.session.timeZone", "UTC")


def table(suffix: str) -> str:
    name = f"{prefix}_{suffix}"
    return f"{database}.{name}" if database else name


missing_tables = [
    table(suffix)
    for suffix in RESET_ORDER
    if not spark_session.catalog.tableExists(table(suffix))
]
missing_views = [
    table(suffix)
    for suffix in COMMITTED_VIEWS
    if not spark_session.catalog.tableExists(table(suffix))
]
if missing_tables or missing_views:
    raise RuntimeError(
        f"Reset target is incomplete; missing tables={missing_tables}, "
        f"missing views={missing_views}"
    )

schemas_before = {
    suffix: spark_session.table(table(suffix)).schema.json()
    for suffix in RESET_ORDER
}
rows_before = {
    suffix: spark_session.table(table(suffix)).count()
    for suffix in RESET_ORDER
}

for suffix in RESET_ORDER:
    DeltaTable.forName(spark_session, table(suffix)).delete()

epoch = datetime(1970, 1, 1, tzinfo=timezone.utc)
registration_table = table("registration_leases")
registration_seed = spark_session.createDataFrame(
    [("global", "", epoch, epoch)],
    schema=spark_session.table(registration_table).schema,
)
(
    DeltaTable.forName(spark_session, registration_table)
    .alias("t")
    .merge(registration_seed.alias("s"), "t.lock_name = s.lock_name")
    .whenNotMatchedInsertAll()
    .execute()
)

schema_changes = [
    suffix
    for suffix in RESET_ORDER
    if spark_session.table(table(suffix)).schema.json() != schemas_before[suffix]
]
unexpected_rows = {}
for suffix in RESET_ORDER:
    actual = spark_session.table(table(suffix)).count()
    expected = 1 if suffix == "registration_leases" else 0
    if actual != expected:
        unexpected_rows[suffix] = {"expected": expected, "actual": actual}

nonempty_views = {
    suffix: spark_session.table(table(suffix)).count()
    for suffix in COMMITTED_VIEWS
    if spark_session.table(table(suffix)).limit(1).count() != 0
}
if schema_changes or unexpected_rows or nonempty_views:
    raise RuntimeError(
        f"Reset verification failed; schema_changes={schema_changes}, "
        f"unexpected_rows={unexpected_rows}, nonempty_views={nonempty_views}"
    )

outcome = {
    "target": target,
    "environment": environment,
    "deleted_rows": sum(rows_before.values()),
    "registration_seed_rows": 1,
    "tables_verified": len(RESET_ORDER),
    "views_verified": len(COMMITTED_VIEWS),
}
notebookutils.notebook.exit(json.dumps(outcome, sort_keys=True))